# UMAP Video Generator

Generate combined videos that show a dataset view alongside the saved UMAP watershed density map, with a red marker indicating the current classified position.

Use the configuration cell below to change dataset selection, paths, and prototype settings. Leave the implementation cells unchanged unless you are modifying the rendering logic.

## User Configuration

Edit this section only if you want to change notebook inputs or output behavior.

What you can change here:
- `DATASET_SELECTION` to choose `hive1_upper`, `hive2_upper`, `hive2_lower`, or `all`.
- `PROTOTYPE_FRAME_LIMIT` to control how many source-video frames are processed during prototyping.
- `LAYOUT`, `FPS`, and output paths if you want a different video layout or destination.
- The `video_alias` entries if your alias-backed source videos move.


In [48]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
RESULTS_DIR = PROJECT_DIR / "Results" / "WideNormalisation" / "UMAP"
WSHED_PATH = RESULTS_DIR / "zVals_wShed_groups.mat"
OUTPUT_DIR = PROJECT_DIR / "outputVideos" / "umap_video"
LAYOUT = "side_by_side"  # or "top_bottom"
FPS = 20
VIDEO_PREFIX = "dataset_umap"
FRAME_LIMIT = None
BUCKET = "ObsHiveABC"
DOWNLOAD_RESOLUTION = 600
SOURCE_VIDEO_SCALE = 0.4
UMAP_PANEL_FIGSIZE = (8, 8)
UMAP_PANEL_DPI = 180
UMAP_PANEL_TITLE_FONTSIZE = 16
UMAP_PANEL_AXIS_LABELSIZE = 14
UMAP_PANEL_TICK_LABELSIZE = 12

# Dataset selection and alias-backed source videos
DATASET_CONFIGS = {
    "hive1_upper": {
        "hive_nb": 1,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-09-04 01:00:00').tz_localize("UTC"),
        "end_ts": pd.Timestamp('2024-10-29 23:00:00').tz_localize("UTC"),
        "umap_row_offset": 12,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset2",
    },
    "hive2_upper": {
        "hive_nb": 2,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-10-03 01:00:00').tz_localize("UTC"),
        "end_ts": pd.Timestamp('2024-10-29 23:00:00').tz_localize("UTC"),
        "umap_row_offset": 0,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
    "hive2_lower": {
        "hive_nb": 2,
        "ihl": "lower",
        "start_ts": pd.Timestamp('2024-10-01 01:00:00').tz_localize("UTC"),
        "end_ts": pd.Timestamp("2024-10-29 23:00:00").tz_localize("UTC"),
        "umap_row_offset": 0,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
}

DATASET_SELECTION = "all"  # choose one key above or "all"


## Import Required Libraries

Import libraries for data handling, plotting, image processing, and video creation. The helper module provides the reusable UMAP/video frame functions.

In [46]:
import sys
import importlib
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from tqdm.auto import tqdm
import imageio.v2 as imageio

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import umap_video_utils as umap_video_utils_module
umap_video_utils_module = importlib.reload(umap_video_utils_module)

compose_panels = umap_video_utils_module.compose_panels
load_umap_points_for_source = umap_video_utils_module.load_umap_points_for_source
load_watershed_artifacts = umap_video_utils_module.load_watershed_artifacts
render_umap_panel = umap_video_utils_module.render_umap_panel

from InfluxDBInterface.libdb import download_tmp_DB, download_data_DB, download_co2_DB
from HiveOpenings.libOpenings import filter_timestamps

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Load Video Frames, Thermal Data, and Metabolic Data

Load the source dataframe for each dataset and keep timestamps aligned across sources.

In [58]:
def download_dataset(hive_nb: int, ihl: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp, resolution: int = DOWNLOAD_RESOLUTION) -> pd.DataFrame:
    """Download the dataset for a given hive and in-hive location using the existing InfluxDB helpers."""
    assert ihl in ["upper", "lower"], "inhive_loc must be either 'upper' or 'lower'"
    assert hive_nb in [1, 2], "hive_num must be either 1 or 2"

    filters = {"hive_num": hive_nb, "inhive_loc": ihl}
    df = download_tmp_DB(BUCKET, start_ts, end_ts, resolution=resolution, filters=filters, aggr="last")

    filters.pop("inhive_loc")
    co2_data = download_co2_DB(BUCKET, start_ts, end_ts, resolution=resolution, filters=filters)
    co2_data = co2_data.loc[:, co2_data.columns.str.startswith(ihl[0])]
    for col in co2_data.columns:
        col_name = f"co2_{col[-1].upper()}"
        df[col_name] = co2_data.loc[:, col]

    filters["field"] = ["rel_humid"]
    filters["measurement"] = ["co2", "rht"]
    filters["inhive_loc"] = ihl
    humid_data = download_data_DB(BUCKET, start_ts, end_ts, resolution=resolution, filters=filters)
    df["rel_humid"] = df.index.to_series().apply(lambda ts: humid_data.loc[humid_data.index == ts, "_value"].mean())

    print("Before filtering with HiveOpenings:", len(df), "lines")
    filtered_ts = filter_timestamps(df.index.to_list(), hive_nb=hive_nb, recovery_time=240)
    df_resampled = df[df.index.isin(filtered_ts)].copy()
    print("After  :", len(df_resampled), "lines")

    df_resampled = df_resampled.sort_index()
    df_resampled["source_id"] = f"hive{hive_nb}_{ihl}"
    df_resampled["real_timestamp"] = df_resampled.index
    return df_resampled


def select_datasets(selection: str):
    if selection == "all":
        return list(DATASET_CONFIGS.keys())
    if selection not in DATASET_CONFIGS:
        raise ValueError(f"Unknown dataset selection: {selection}")
    return [selection]


selected_datasets = select_datasets(DATASET_SELECTION)
loaded_datasets = {
    name: download_dataset(
        hive_nb=DATASET_CONFIGS[name]["hive_nb"],
        ihl=DATASET_CONFIGS[name]["ihl"],
        start_ts=DATASET_CONFIGS[name]["start_ts"],
        end_ts=DATASET_CONFIGS[name]["end_ts"],
        resolution=DOWNLOAD_RESOLUTION,
    )
    for name in selected_datasets
}
loaded_datasets


Before filtering with HiveOpenings: 7728 lines
After  : 7244 lines
Before filtering with HiveOpenings: 3571 lines
After  : 3391 lines
Before filtering with HiveOpenings: 3704 lines
After  : 3491 lines


{'hive1_upper':                               t00        t01        t02        t03        t04        t05        t06        t07     \
 datetime                                                                                                            
 2024-09-04 14:00:00+00:00  32.492188  32.453125  32.507812  32.531250  32.601562  32.781250  32.820312  32.265625   
 2024-09-04 14:10:00+00:00  32.609375  32.617188  32.796875  32.984375  32.742188  32.835938  32.937500  32.382812   
 2024-09-04 14:20:00+00:00  32.890625  32.906250  33.148438  33.421875  32.859375  32.929688  33.125000  32.820312   
 2024-09-04 14:30:00+00:00  33.234375  33.125000  33.414062  33.578125  33.054688  33.078125  33.367188  33.289062   
 2024-09-04 14:40:00+00:00  33.484375  33.273438  33.570312  33.695312  33.156250  33.257812  33.757812  33.851562   
 ...                              ...        ...        ...        ...        ...        ...        ...        ...   
 2024-10-28 08:00:00+00:00  22.289062  22

## Load UMAP Embedding, Density Map, and Classifier Outputs

Load the precomputed watershed density background and the border coordinates needed to place the current timestamp on the UMAP map.

In [35]:
import importlib
import umap_video_utils as umap_video_utils_module
importlib.reload(umap_video_utils_module)
load_watershed_artifacts = umap_video_utils_module.load_watershed_artifacts
wshed_artifacts = load_watershed_artifacts(str(WSHED_PATH))
wshed_artifacts


{'density': array([[2.58049852e-06, 2.79656072e-06, 3.03059574e-06, ...,
         2.02959337e-06, 2.19801133e-06, 2.38132386e-06],
        [2.82131964e-06, 3.05954761e-06, 3.31730416e-06, ...,
         2.21230152e-06, 2.39878451e-06, 2.60143815e-06],
        [3.08674492e-06, 3.34910292e-06, 3.63270499e-06, ...,
         2.41459258e-06, 2.62067733e-06, 2.84434315e-06],
        ...,
        [1.98933871e-06, 2.14913883e-06, 2.32310204e-06, ...,
         1.58675539e-06, 1.70891241e-06, 1.84285996e-06],
        [2.16601173e-06, 2.34300628e-06, 2.53530024e-06, ...,
         1.71793509e-06, 1.85431196e-06, 2.00340088e-06],
        [2.36258465e-06, 2.55828587e-06, 2.77055343e-06, ...,
         1.86519349e-06, 2.01695112e-06, 2.18245306e-06]]),
 'xx': array([[-8.99956665e+01, -8.97005987e+01, -8.94055310e+01,
         -8.91104632e+01, -8.88153955e+01, -8.85203277e+01,
         -8.82252599e+01, -8.79301922e+01, -8.76351244e+01,
         -8.73400567e+01, -8.70449889e+01, -8.67499212e+01,
        

## Build Timestamp-to-UMAP Coordinate Lookup

Create reusable functions that map each frame timestamp to the corresponding UMAP coordinate for the selected dataset.

In [20]:
def build_timestamp_lookup(df: pd.DataFrame, umap_points: np.ndarray, timestamp_col: str = "real_timestamp") -> pd.DataFrame:
    if timestamp_col in df.columns:
        timestamps = pd.to_datetime(df[timestamp_col])
    else:
        timestamps = pd.to_datetime(df.index)
    coords = np.asarray(umap_points, dtype=float)
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError("UMAP coordinates must be an (N, 2) array")
    if len(coords) != len(df):
        raise ValueError("UMAP coordinates and dataframe must have the same length")
    lookup = pd.DataFrame({"timestamp": timestamps, "umap_x": coords[:, 0], "umap_y": coords[:, 1]})
    if "region" in df.columns:
        lookup["region"] = pd.to_numeric(df["region"], errors="coerce").to_numpy()
    if "source_id" in df.columns:
        lookup["source_id"] = df["source_id"].astype(str).to_numpy()
    return lookup


def resolve_point_for_timestamp(lookup: pd.DataFrame, timestamp) -> np.ndarray:
    ts = pd.Timestamp(timestamp)
    row = lookup.loc[lookup["timestamp"] == ts]
    if row.empty:
        return lookup[["umap_x", "umap_y"]].iloc[0].to_numpy(dtype=float)
    return row[["umap_x", "umap_y"]].iloc[0].to_numpy(dtype=float)


## Create Frame Overlay for the Current Classification Point

Draw a red dot on the UMAP density map at the classified position for the current frame and prepare the visualization for frame-by-frame updates.

In [36]:
def render_umap_frame_for_row(row: pd.Series, lookup: pd.DataFrame, *, wshed_artifacts: dict, timestamp_col: str = "real_timestamp") -> np.ndarray:
    timestamp = row[timestamp_col] if timestamp_col in row.index else row.name
    point = resolve_point_for_timestamp(lookup, timestamp)
    region = row["region"] if "region" in row.index else None
    return render_umap_panel(
        point,
        wshed_artifacts["density"],
        extent=wshed_artifacts["extent"],
        wbounds=wshed_artifacts["wbounds"],
        barycenters=None,
        title="UMAP density map",
        timestamp=timestamp,
        region=region,
        figsize=UMAP_PANEL_FIGSIZE,
        dpi=UMAP_PANEL_DPI,
        title_fontsize=UMAP_PANEL_TITLE_FONTSIZE,
        axis_labelsize=UMAP_PANEL_AXIS_LABELSIZE,
        tick_labelsize=UMAP_PANEL_TICK_LABELSIZE,
    )


## Compose Side-by-Side or Top-Bottom Video Layouts

Implement layout functions to place the main video content and the UMAP panel either side by side or stacked vertically.

In [57]:
def blackout_unused_hive_half(dataset_frame: np.ndarray, ihl: str) -> np.ndarray:
    frame = np.asarray(dataset_frame).copy()
    if frame.ndim != 3 or frame.shape[0] < 2 or frame.shape[1] < 2:
        return frame

    original = frame.copy()
    height = frame.shape[0]
    width = frame.shape[1]
    midpoint = height // 2

    if ihl == "upper":
        frame[midpoint:, :, :] = 0
        ambient_x0 = int(width * (2700 / 3840))
        ambient_y0 = int(height * (2050 / 2160))
        frame[ambient_y0:height, ambient_x0:width, :] = original[ambient_y0:height, ambient_x0:width, :]
    elif ihl == "lower":
        frame[:midpoint, :, :] = 0
        timestamp_x0 = int(width * (1500 / 3840))
        timestamp_y0 = int(height * (980 / 2160))
        timestamp_x1 = int(width * (2350 / 3840))
        timestamp_y1 = int(height * (1165 / 2160))
        frame[timestamp_y0:timestamp_y1, timestamp_x0:timestamp_x1, :] = original[timestamp_y0:timestamp_y1, timestamp_x0:timestamp_x1, :]
    return frame


## Render Combined Video for a Single Dataset

Assemble the video frames with the UMAP panel, write the final video to disk, and expose this as a function that accepts a dataset identifier.

In [ ]:
def resolve_macos_alias(path: str) -> Path:
    source_path = Path(path)
    if source_path.is_symlink():
        return source_path.resolve()
    apple_script = (
        'set a to POSIX file "{}" as alias\n'
        'tell application "Finder" to set b to original item of a\n'
        'return POSIX path of (b as alias)'
    ).format(str(source_path).replace('"', '\\"'))
    try:
        resolved = subprocess.check_output(["osascript", "-e", apple_script], universal_newlines=True).strip()
        if resolved:
            return Path(resolved)
    except Exception:
        pass
    return source_path


def render_dataset_video(df: pd.DataFrame, dataset_name: str, *, layout: str = LAYOUT, fps: int = FPS) -> str:
    dataset_config = DATASET_CONFIGS[dataset_name]
    row_offset = dataset_config.get("umap_row_offset", 0)

    ordered_dataset_names = selected_datasets if 'selected_datasets' in globals() else list(DATASET_CONFIGS.keys())
    dataset_offset = 0
    if 'loaded_datasets' in globals():
        for name in ordered_dataset_names:
            if name == dataset_name:
                break
            dataset_offset += len(loaded_datasets[name])

    umap_points_all = load_umap_points_for_source(str(WSHED_PATH), dataset_name)
    available_points = max(0, len(umap_points_all) - dataset_offset)
    source_video_path = resolve_macos_alias(dataset_config["video_alias"] )
    df_available = max(0, len(df) - row_offset)
    umap_available = max(0, available_points - row_offset)
    cap = cv2.VideoCapture(str(source_video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    n_points = min((df_available + 1) // 2, (umap_available + 1) // 2, frame_count)
    if FRAME_LIMIT is not None:
        n_points = min(n_points, FRAME_LIMIT)
    if len(df) != available_points:
        print(f"Warning: aligning {dataset_name} to {n_points} rows using offset {dataset_offset} ({len(df)} dataframe rows, {available_points} UMAP points in this segment).")
    df = df.iloc[row_offset:row_offset + n_points * 2:2].copy()
    umap_points = np.asarray(umap_points_all, dtype=float)[dataset_offset + row_offset:dataset_offset + row_offset + n_points * 2:2]
    lookup = build_timestamp_lookup(df, umap_points)
    reader = imageio.get_reader(str(source_video_path))

    def frame_iterator():
        progress = tqdm(total=n_points, desc=f'"{dataset_name}" video frames', unit='frame')
        for video_frame in reader:
            dataset_frame = np.asarray(video_frame)
            if dataset_frame.ndim == 2:
                dataset_frame = np.stack([dataset_frame] * 3, axis=-1)
            if dataset_frame.shape[2] == 4:
                dataset_frame = dataset_frame[:, :, :3]
            if SOURCE_VIDEO_SCALE != 1.0:
                new_width = max(1, int(dataset_frame.shape[1] * SOURCE_VIDEO_SCALE))
                new_height = max(1, int(dataset_frame.shape[0] * SOURCE_VIDEO_SCALE))
                dataset_frame = cv2.resize(dataset_frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
            dataset_frame = blackout_unused_hive_half(dataset_frame, dataset_config["ihl"] )
            row_index = progress.n
            if row_index >= len(df):
                break
            row = df.iloc[row_index]
            umap_frame = render_umap_frame_for_row(row, lookup, wshed_artifacts=wshed_artifacts)
            yield compose_panels(dataset_frame, umap_frame, layout=layout)
            progress.update(1)
        progress.close()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generate_video_name = f"{VIDEO_PREFIX}_{dataset_name}"
    output_path = generateVideoFromFrames(frame_iterator(), dest=str(OUTPUT_DIR), name=generate_video_name, fps=fps)
    reader.close()
    return output_path


## Loop Over the Selected Dataset

Add batch-processing logic to generate videos for each dataset using the same rendering pipeline and configurable settings.

In [ ]:
video_outputs = {}
for dataset_name, dataset_df in loaded_ datasets.items():
    video_outputs[dataset_name] = render_dataset_video(dataset_df, dataset_name, layout=LAYOUT, fps=FPS)

video_outputs


"hive1_upper" video frames:   0%|          | 0/3616 [00:00<?, ?frame/s]

"hive2_upper" video frames:   0%|          | 0/1696 [00:00<?, ?frame/s]

"hive2_lower" video frames:   0%|          | 0/451 [00:00<?, ?frame/s]

{'hive1_upper': '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive1_upper_1.mp4',
 'hive2_upper': '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive2_upper_1.mp4',
 'hive2_lower': '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive2_lower_2.mp4'}

## Verify Generated Videos

Confirm that the files were created successfully.

In [55]:
verified_outputs = {name: Path(path).exists() for name, path in video_outputs.items()}
verified_outputs

{'hive1_upper': True, 'hive2_upper': True, 'hive2_lower': True}

In [66]:
hive1_upper_output = render_dataset_video(loaded_datasets['hive1_upper'], 'hive1_upper', layout=LAYOUT, fps=FPS)
print(hive1_upper_output)


"hive1_upper" video frames:   0%|          | 0/3874 [00:00<?, ?frame/s]

/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive1_upper_1.mp4


KeyboardInterrupt: 